In [2]:
import numpy as np
import sys
sys.path.append('../../../../Constrictivity/William_Holzer/intrustion_const/')
sys.path.append('../../../../Segmentation/Will_3D_MICRO_FAST/')
from Intrusion import get_constrictivity, calc_iPSD, find_median_reversed
from size_dist import size_dist_basic
import basic_microstruct_analysis as bma

In [3]:
import glob

In [6]:
files = glob.glob('../../data_new/Fine*/segmented_percolated.npy')
files = sorted(files)

In [8]:
phases = [3]

In [15]:
rows = []

for file in files:
    data_file = np.load(file)
    data_x = np.transpose(data_file, (2, 0, 1))   # transport direction -> axis 0, for BOTH τ and β

    for phase in phases:
        t0 = time.perf_counter()

        mask_x = (data_x == phase).astype(np.uint8)

        tau_factor = bma.tortuosity_factor(data_x, ph=phase, axis=0)   # = tau_geom**2

        beta_x, cPSD, iPSD, diams, rmin, rmax = get_constrictivity(
            mask_x, phase, fast=True, ret_dist=True, num_faces=1
        )

        phi = mask_x.mean()
        sigma_eff = phi * beta_x / tau_factor          # = phi * beta / tau_geom**2

        rows.append({
            "file": os.path.basename(os.path.dirname(file)),
            "phase": phase,
            "tau_geom": np.sqrt(tau_factor),
            "tau_factor": tau_factor,
            "beta_x": beta_x,
            "rmin": rmin, "rmax": rmax,
            "phi": phi,
            "sigma_eff": sigma_eff,                    # no rounding
            "time_seconds": time.perf_counter() - t0,  # per phase
        })

results_df = pd.DataFrame(rows)
results_df

,file,phase,tau_geom,tau_factor,beta_x,rmin,rmax,phi,sigma_eff,time_seconds
0,Fine_microstructure_0,3,1.028687,1.058198,0.459799,11.391222,16.799109,0.466372,0.202644,15.770685
1,Fine_microstructure_1,3,1.035624,1.072516,0.464053,10.469454,15.368815,0.424495,0.183669,20.854077
2,Fine_microstructure_2,3,1.031368,1.063721,0.341898,9.084028,15.535674,0.403174,0.129587,12.771571
3,Fine_microstructure_3,3,1.048980,1.100360,0.549610,10.930026,14.743273,0.386213,0.192907,23.064201
4,Fine_microstructure_4,3,1.023086,1.046704,0.435193,10.123059,15.345139,0.428214,0.178041,18.694093
5,Fine_microstructure_5,3,1.045187,1.092415,0.484622,10.500614,15.083879,0.388813,0.172487,20.325544
6,Fine_microstructure_6,3,1.027497,1.055751,0.428871,9.812051,14.982922,0.416761,0.169298,14.727100
7,Fine_microstructure_7,3,1.038080,1.077609,0.342965,8.351569,14.260781,0.400439,0.127445,21.552869
8,Fine_microstructure_8,3,1.042805,1.087442,0.450130,10.096600,15.048946,0.402074,0.166432,16.819687
9,Fine_microstructure_9,3,1.042380,1.086556,0.464324,9.902885,14.532870,0.410423,0.175388,16.262241


In [16]:
results_df.to_csv("../Holzer_Method_results.csv", index=False, float_format="%.4f")